# Kiểm demo live M1

Chạy sau khi GPU không còn làm validation. Kiểm ba ảnh validation ngoài gallery, nạp lại model và lưu đầu ra/thời gian/VRAM. Không train, không mở test hoặc công khai server. Đây là kiểm kỹ thuật, không phải đánh giá chất lượng.

In [ ]:
import hashlib
import json
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = os.environ.get('VQA_PROJECT_ROOT')
SOURCE_RELEASE = ''  # Bản source đã kiểm có notebook 06
SOURCE_SHA256 = ''
RUN_NAME = 'm1_live_check_01'
RUN_DEMO_CHECK = False
PLAN = None


## 1. Drive và môi trường

Source và adapter chỉ đọc. Kết quả ghi vào thư mục mới trong `demo_runs`.

In [ ]:
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR).resolve()
else:
    candidates = [Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob('*/Vietnamese_VQA_NLE'))
    roots = {p.resolve() for p in candidates
             if (p / 'reports/m1_selection/frozen_checkpoint3683_v2/selection.json').is_file()}
    if len(roots) != 1:
        raise ValueError('Set PROJECT_DIR to the mounted project folder')
    ROOT = roots.pop()
os.environ['VQA_PROJECT_ROOT'] = str(ROOT)
print('Project:', ROOT)


In [ ]:
PLAN = None
if not re.fullmatch(r'source_[A-Za-z0-9_]+', SOURCE_RELEASE or ''):
    raise ValueError('Select a verified source release')
if not re.fullmatch(r'[0-9a-f]{64}', SOURCE_SHA256 or ''):
    raise ValueError('Select its SHA-256')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', RUN_NAME):
    raise ValueError('Use a simple new run name')
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
):
    raise ValueError('Bootstrap checksum mismatch')
REPO = Path(subprocess.check_output([
    sys.executable, str(bootstrap), '--archive',
    str(ROOT / 'source/releases' / f'{SOURCE_RELEASE}.zip'),
    '--sha256', SOURCE_SHA256, '--work-root', '/content/vqa_sources',
], text=True).strip())
run_command = runpy.run_path(str(REPO / 'src/vqa_explanation/utils/process.py'))['run_command']
def run(command):
    run_command([str(part) for part in command], cwd=REPO)
ENV_DIR = Path('/content/vqa_envs') / f'{SOURCE_RELEASE}_demo'
PYTHON = ENV_DIR / 'bin/python'
run([sys.executable, REPO / 'scripts/setup_cloud_environment.py',
     '--profile', 'demo', '--environment-dir', ENV_DIR])
print('Source:', SOURCE_RELEASE, 'Python:', PYTHON)


In [ ]:
PLAN = None
WORK_DATA = Path('/content/vqa_work/vivqax_image_disjoint_v1')
bundle = ROOT / 'datasets/releases/vivqax_image_disjoint_v1'
if hashlib.sha256((bundle / 'bundle.json').read_bytes()).hexdigest() != (
    '46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522'
):
    raise ValueError('Dataset bundle changed')
if not (WORK_DATA / 'metadata/all.jsonl').is_file():
    run([PYTHON, REPO / 'scripts/verify_coco_bundle.py', bundle, '--extract-to', WORK_DATA])
os.environ['VQA_WORK_DATA'] = str(WORK_DATA)
COMMAND = [
    str(PYTHON), '-u', str(REPO / 'scripts/check_demo_live.py'),
    '--config', str(REPO / 'configs/demo/m1_colab.example.yaml'),
    '--environment', str(REPO / 'configs/environments/demo_colab.example.yaml'),
    '--adapter', str(ROOT / 'runs/m1/sft-full_20261007_023531_870900/training/checkpoint-3683'),
    '--output-dir', str(ROOT / 'demo_runs' / RUN_NAME),
]


## 2. Kiểm đầu vào và chạy GPU

Inspect không nạp model hoặc tạo kết quả. Chỉ bật `RUN_DEMO_CHECK` sau khi kiểm đúng adapter, ba ảnh và thư mục output.

In [ ]:
PLAN = None
child_env = {k: v for k, v in os.environ.items() if k not in {'PYTHONPATH', 'PYTHONHOME'}}
child_env['PYTHONNOUSERSITE'] = '1'
PLAN = json.loads(subprocess.check_output(
    COMMAND + ['--inspect'], cwd=REPO, text=True, env=child_env))
print(json.dumps(PLAN, ensure_ascii=False, indent=2))


In [ ]:
if PLAN is None:
    raise RuntimeError('Inspect demo inputs first')
if RUN_DEMO_CHECK:
    run(COMMAND + ['--expected-plan-sha256', PLAN['sha256']])
else:
    print('GPU demo check not requested')


## 3. Kết quả

`technical_passed` chỉ xác nhận luồng kỹ thuật. Vẫn cần xem giao diện trên trình duyệt và thử ảnh ngoài bộ dữ liệu trước nghiệm thu demo cuối.

In [ ]:
from IPython.display import Image, display

report_path = ROOT / 'demo_runs' / RUN_NAME / 'report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text('utf-8'))
    print('Status:', report['status'], 'GPU:', report['gpu'])
    print('Peak allocated bytes:', report.get('peak_allocated_bytes'))
    for row in report['cases']:
        display(Image(filename=str(WORK_DATA / row['image_path'])))
        print(row['question'])
        print(row['result']['raw_output'])
        print('Seconds:', row['request_seconds'])
    print('Report:', report_path)
else:
    print('Demo live check not completed')
